In [ ]:
!pip install -q transformers torch scikit-learn pandas numpy matplotlib seaborn
# Vérification GPU
import torch
print("GPU disponible :", torch.cuda.is_available())
print("Device :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


GPU disponible : False
Device : CPU


In [ ]:
import re
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from transformers import (
    CamembertTokenizer,
    CamembertModel,
    CamembertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW # Corrected import path for AdamW
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Détection automatique GPU/CPU
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM disponible : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device utilisé : cpu


In [ ]:
#importer les donnees
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))


In [ ]:
# =============================================================================
# CELLULE 3 — Chargement des données
# =============================================================================

df = pd.read_csv('corpus_haineux.csv', sep=';', quotechar='"')
df.columns = df.columns.str.strip()
df['tweet'] = df['tweet'].astype(str)

print(f"Tweets chargés : {len(df)}")
print(df['label'].value_counts().rename({0: 'Non haineux', 1: 'Haineux'}))

In [ ]:
# =============================================================================
# CELLULE 4 — Prétraitement adapté à CamemBERT
# =============================================================================
#
#  RÈGLES IMPORTANTES pour CamemBERT (différent de Naïve Bayes et les autres Ml traditionnels ) :
#
#  --->  NE PAS supprimer les stopwords  → CamemBERT comprend le contexte
#     grammatical, "ne ... pas" ensemble a un sens différent de séparé
#
#  --->  NE PAS lemmatiser               → CamemBERT a appris les flexions
#     ("envahissent" vs "envahir" sont différents dans le contexte)
#
#  --->  Remplacer URLs par [URL]        → le tokenizer ne sait pas quoi faire
#     d'un lien, mais [URL] lui dit "il y avait un lien ici"
#
#  --->  Remplacer @pseudo par @user     → anonymise les mentions tout en
#     conservant le signal "quelqu'un a été tagué"
#
#  --->  Garder les emojis               → très informatifs pour la haine
#
#
#  --->  Garder la casse                 → CamemBERT est "cased" : il a appris
#     que "CONNARDS" ≠ "connards" (les majuscules = colère/emphase)
#
#  --->  Garder les retours à la ligne réduits → CamemBERT gère le contexte
#
#  --->  Supprimer "RT" → marqueur technique sans valeur sémantique

def preprocess_camembert(text: str) -> str:
    """
    Prétraitement minimal adapté à CamemBERT.
    On nettoie le bruit technique sans toucher au contenu linguistique.
    """
    # 1. Remplacer URLs par token générique
    text = re.sub(r'http\S+|www\S+', '[URL]', text)

    # 2. Remplacer @mentions par @user (anonymisation + conservation du signal)
    text = re.sub(r'@\w+', '@user', text)

    # 3. Supprimer RT (marqueur de retweet, bruit pur)
    text = re.sub(r'\bRT\b', '', text)

    # 4. Réduire les espaces multiples / retours à la ligne
    text = re.sub(r'\s+', ' ', text).strip()

    # NE PAS : supprimer stopwords, lemmatiser, mettre en lowercase, supprimer emojis, supprimer ponctuation

    return text


df['tweet_clean'] = df['tweet'].apply(preprocess_camembert)

print("\nExemples avant / après prétraitement CamemBERT :")
for _, row in df.sample(3, random_state=7).iterrows():
    label = "HAINEUX" if row['label'] == 1 else "NON HAINEUX"
    print(f"\n  [{label}]")
    print(f"  Avant : {row['tweet'][:100].strip()}")
    print(f"  Après : {row['tweet_clean'][:100].strip()}")

In [ ]:
# =============================================================================
# VERSION 1 — SANS FINE-TUNING (Feature Extraction)
# =============================================================================
#
#  Principe :
#  1. On charge CamemBERT avec ses poids pré-entraînés (gelés, on ne les modifie pas)
#  2. On passe chaque tweet dans CamemBERT
#  3. On récupère le vecteur du token <s> (premier token, équivalent du [CLS] de BERT)
#     → Ce vecteur de 768 dimensions "résume" le tweet entier
#  4. On entraîne une Régression Logistique simple sur ces vecteurs
#
#  POURQUOI Régression Logistique et pas SVM ?
#  → Les embeddings sont denses (768 dims pleines, pas sparse)
#  → LR converge plus vite sur Colab +  donne des probabilités, pas juste des classes
#  → SVM avec RBF sur 768 dims = très lent et pas nécessairement meilleur + RL a donné the best resultats pour methodes tradi


In [ ]:
#Dédupliquer avant la validation croisée
df['tweet'] = df['tweet'].astype(str)

In [ ]:
# Supprimer les doublons exacts tweet + label
before = len(df)
df = df.drop_duplicates(subset=['tweet', 'label']).reset_index(drop=True)
after = len(df)

print(f"Doublons supprimés : {before - after}")

In [ ]:
print(f"Tweets chargés : {len(df)}")
print(df['label'].value_counts().rename({0: 'Non haineux', 1: 'Haineux'}))

In [ ]:
# =============================================================================
# CELLULE 6 — Extraction des embeddings (Version 1)
# =============================================================================

print("\n" + "=" * 55)
print("VERSION 1 — Extraction des embeddings CamemBERT")
print("=" * 55)

MODEL_NAME = 'camembert-base'
TOKENIZER = CamembertTokenizer.from_pretrained(MODEL_NAME)
MAX_LEN = 128

camembert_extractor = CamembertModel.from_pretrained(MODEL_NAME)
camembert_extractor.eval()
camembert_extractor.to(DEVICE)

for param in camembert_extractor.parameters():
    param.requires_grad = False

def mean_pooling(last_hidden_state, attention_mask):
    """
    Mean pooling en ignorant les tokens de padding.
    """
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    masked_embeddings = last_hidden_state * mask
    summed = masked_embeddings.sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

def get_sentence_embedding(texts, batch_size=32):
    """
    Retourne les embeddings moyens des tokens non-padding.
    """
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]

        encoded = TOKENIZER(
            batch_texts,
            max_length=MAX_LEN,
            padding=True,
            truncation=True,
            return_tensors='pt'
        )

        input_ids = encoded['input_ids'].to(DEVICE)
        attention_mask = encoded['attention_mask'].to(DEVICE)

        with torch.no_grad():
            outputs = camembert_extractor(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        sentence_embeddings = mean_pooling(outputs.last_hidden_state, attention_mask)
        all_embeddings.append(sentence_embeddings.cpu().numpy())

        if (i // batch_size) % 5 == 0:
            print(f"  Batch {i // batch_size + 1}/{len(texts) // batch_size + 1} traité")

    return np.vstack(all_embeddings)

texts_list = df['tweet_clean'].tolist()
print("Extraction en cours (peut prendre 1-2 min sur Colab GPU)...")
X_embeddings = get_sentence_embedding(texts_list, batch_size=32)
y = df['label'].values

print(f"\nEmbeddings extraits : {X_embeddings.shape}")

del camembert_extractor
torch.cuda.empty_cache()

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# =============================================================================
# CELLULE 7 — Classification avec Régression Logistique + CV (Version 1)
# =============================================================================

print("\n" + "=" * 55)
print("VERSION 1 — Régression Logistique sur embeddings")
print("=" * 55)

#  max_iter=1000 : embeddings de haute dim nécessitent plus d'itérations
#  C=1.0 : régularisation standard (ni trop forte, ni trop faible)
#  solver='lbfgs' : optimal pour données denses de haute dimension

lr_classifier = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=1.0,
        max_iter=2000,
        solver='lbfgs',
        random_state=42
    )
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# cross_val_predict → prédictions sur les 600 tweets (cohérent, pas de fuite)
y_pred_v1 = cross_val_predict(lr_classifier, X_embeddings, y, cv=cv)

print("\nRÉSULTATS — Version 1 (sans fine-tuning) :")
print(f"  Accuracy  = {accuracy_score(y, y_pred_v1):.4f}")
print(f"  Precision = {precision_score(y, y_pred_v1, average='macro'):.4f}")
print(f"  Recall    = {recall_score(y, y_pred_v1, average='macro'):.4f}")
print(f"  F1        = {f1_score(y, y_pred_v1, average='macro'):.4f}")

# Matrice de confusion Version 1
cm_v1 = confusion_matrix(y, y_pred_v1)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_v1, display_labels=['Non haineux', 'Haineux']).plot(
    ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matrice de confusion — CamemBERT sans fine-tuning\n(cross_val_predict, k=5)', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_v1_camembert.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRapport détaillé :")
print(classification_report(y, y_pred_v1, target_names=['Non haineux', 'Haineux']))


In [ ]:
# =============================================================================
# VERSION 2 — AVEC FINE-TUNING
# =============================================================================
#
#  Principe :
#  On ajoute une couche de classification (Linear 768 → 2) par-dessus CamemBERT
#  et on met à jour TOUS les poids (CamemBERT + classifieur) pendant l'entraînement.
#  CamemBERT s'adapte alors spécifiquement au vocabulaire haineux de tes tweets.
#
#  CHOIX DES HYPERPARAMÈTRES (justifiés par notre dataset 600 tweets) :
#
#  • Learning Rate = 2e-5  → très faible pour ne pas "écraser" les poids
#                             pré-entraînés (catastrophic forgetting)
#                             5e-5 serait risqué sur 600 tweets seulement
#
#
#  • Warmup = 10% des steps → évite les grandes mises à jour au début
#                              (stabilise l'entraînement)

In [ ]:
# =============================================================================
# CELLULE 8 — Dataset PyTorch pour le fine-tuning
# =============================================================================

class TweetDataset(Dataset):
    """Dataset PyTorch pour les tweets tokenisés."""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',   # padding fixe pour le DataLoader
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'label'         : torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
# =============================================================================
# CELLULE 9 — Fonctions d'entraînement et d'évaluation
# =============================================================================

def train_one_epoch(model, dataloader, optimizer, scheduler):
    """Entraîne le modèle sur un epoch complet."""
    model.train()
    total_loss = 0

    for batch in dataloader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels          # CamembertForSequenceClassification calcule
        )                          # la CrossEntropyLoss automatiquement

        loss = outputs.loss
        loss.backward()

        # Gradient clipping : évite les explosions de gradient (bonne pratique BERT)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader):
    """Évalue le modèle et retourne les prédictions + labels."""
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds   = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return np.array(all_labels), np.array(all_preds)


In [ ]:
# =============================================================================
# CELLULE 10 — Fine-Tuning avec Cross-Validation k=5
# =============================================================================

print("\n" + "=" * 55)
print("VERSION 2 — Fine-Tuning CamemBERT (CV k=5)")
print("=" * 55)

# Hyperparamètres
LEARNING_RATE = 2e-5
BATCH_SIZE    = 16
EPOCHS        = 3
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.1    # 10% des steps pour le warmup

texts  = df['tweet_clean'].tolist()
labels = df['label'].values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Stockage des résultats par fold
fold_metrics = {m: [] for m in ['accuracy', 'precision', 'recall', 'f1']}
all_preds_v2  = np.zeros(len(labels), dtype=int)
train_losses_all = []

for fold, (train_idx, test_idx) in enumerate(cv.split(texts, labels)):
    print(f"\n{'─' * 45}")
    print(f"  FOLD {fold + 1} / 5")
    print(f"{'─' * 45}")
    print(f"  Train : {len(train_idx)} tweets | Test : {len(test_idx)} tweets")

    # Données du fold
    train_texts  = [texts[i] for i in train_idx]
    test_texts   = [texts[i] for i in test_idx]
    train_labels = labels[train_idx]
    test_labels  = labels[test_idx]

    # Datasets et DataLoaders
    train_dataset = TweetDataset(train_texts, train_labels, TOKENIZER, MAX_LEN)
    test_dataset  = TweetDataset(test_texts,  test_labels,  TOKENIZER, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

    # Nouveau modèle à chaque fold (important : poids réinitialisés)
    model = CamembertForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2,
        ignore_mismatched_sizes=True
    )
    model.to(DEVICE)

    # Optimizer et scheduler
    total_steps  = len(train_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)

    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    # Entraînement
    fold_losses = []
    for epoch in range(EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        fold_losses.append(loss)
        print(f"  Epoch {epoch + 1}/{EPOCHS} — Loss : {loss:.4f}")

    train_losses_all.append(fold_losses)

    # Évaluation sur le fold test
    true_labels, pred_labels = evaluate(model, test_loader)
    all_preds_v2[test_idx] = pred_labels

    # Métriques du fold
    acc = accuracy_score(true_labels, pred_labels)
    pre = precision_score(true_labels, pred_labels, average='macro', zero_division=0)
    rec = recall_score(true_labels, pred_labels, average='macro', zero_division=0)
    f1  = f1_score(true_labels, pred_labels, average='macro', zero_division=0)

    fold_metrics['accuracy'].append(acc)
    fold_metrics['precision'].append(pre)
    fold_metrics['recall'].append(rec)
    fold_metrics['f1'].append(f1)

    print(f"\n  Résultats fold {fold + 1} :")
    print(f"    Accuracy  = {acc:.4f}")
    print(f"    Precision = {pre:.4f}")
    print(f"    Recall    = {rec:.4f}")
    print(f"    F1        = {f1:.4f}")

    # Libérer mémoire GPU entre les folds
    del model
    torch.cuda.empty_cache()

# Résultats globaux Version 2
print("\n" + "=" * 55)
print("RÉSULTATS FINAUX — Version 2 (avec fine-tuning) :")
print("=" * 55)
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    vals = np.array(fold_metrics[metric])
    print(f"  {metric.capitalize():<12} = {vals.mean():.4f}  (± {vals.std():.4f})")

# Matrice de confusion Version 2
cm_v2 = confusion_matrix(labels, all_preds_v2)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm_v2, display_labels=['Non haineux', 'Haineux']).plot(
    ax=ax, colorbar=False, cmap='Oranges')
ax.set_title('Matrice de confusion — CamemBERT fine-tuné\n(cross_val_predict, k=5)', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_v2_camembert.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nRapport détaillé :")
print(classification_report(labels, all_preds_v2, target_names=['Non haineux', 'Haineux']))


In [ ]:
# =============================================================================
# CELLULE 11 — Courbe de loss par epoch (fine-tuning seulement)
# =============================================================================

print("\n" + "=" * 55)
print("COURBE DE LOSS — Évolution pendant le fine-tuning")
print("=" * 55)

fig, ax = plt.subplots(figsize=(9, 5))
colors = plt.cm.tab10(np.linspace(0, 1, 5))

for fold_idx, losses in enumerate(train_losses_all):
    ax.plot(range(1, EPOCHS + 1), losses, 'o-',
            color=colors[fold_idx], linewidth=2, markersize=6,
            label=f'Fold {fold_idx + 1}')

# Moyenne des folds
mean_losses = np.mean(train_losses_all, axis=0)
ax.plot(range(1, EPOCHS + 1), mean_losses, 's--',
        color='black', linewidth=2.5, markersize=8, label='Moyenne', zorder=5)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Loss (CrossEntropy)', fontsize=12)
ax.set_title('Évolution de la loss d\'entraînement — CamemBERT fine-tuné (k=5)', fontsize=13)
ax.set_xticks(range(1, EPOCHS + 1))
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('loss_finetuning.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# =============================================================================
# CELLULE 13 — Comparaison finale : tous les modèles
# =============================================================================

nb_results = {
    'Accuracy' : 0.914,
    'Precision': 0.915,
    'Recall'   : 0.915,
    'F1'       : 0.913,
}

svm_results = {
    'Accuracy' : 0.926,
    'Precision': 0.925,
    'Recall'   : 0.927,
    'F1'       : 0.925,
}

rl_results = {
    'Accuracy' : 0.929,
    'Precision': 0.928,
    'Recall'   : 0.920,
    'F1'       : 0.92,
}

#v1_results = {
 #   'Accuracy' : accuracy_score(labels, y_pred_v1),
  #  'Precision': precision_score(labels, y_pred_v1, average='macro'),
   # 'Recall'   : recall_score(labels, y_pred_v1, average='macro'),
    #'F1'       : f1_score(labels, y_pred_v1, average='macro'),
#}

v1_results = {
    'Accuracy' : 0.9308,
    'Precision': 0.9307,
    'Recall'   : 0.9307,
    'F1'       : 0.9307,
}


v2_results = {
    'Accuracy' : 0.920,
    'Precision': 0.922,
    'Recall'   : 0.915,
    'F1'       : 0.913,
}

#v2_results = {
  #  'Accuracy' : np.mean(fold_metrics['accuracy']),
   # 'Precision': np.mean(fold_metrics['precision']),
   # 'Recall'   : np.mean(fold_metrics['recall']),
   # 'F1'       : np.mean(fold_metrics['f1']),
#}

# --- Données du graphique ---
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1']
models = {
    'Naïve Bayes'         : (nb_results,  '#4CAF50'),
    'SVM'                 : (svm_results, '#9C27B0'),
    'Régression Logistique': (rl_results,  '#FF9800'),
    'CamemBERT sans FT'   : (v1_results,  '#2196F3'),
    'CamemBERT avec FT'   : (v2_results,  '#FF5722'),
}

n_models  = len(models)
n_metrics = len(metrics_names)
width     = 0.15
x         = np.arange(n_metrics)

fig, ax = plt.subplots(figsize=(14, 7))

for i, (model_name, (model_results, color)) in enumerate(models.items()):
    offset = (i - n_models / 2 + 0.5) * width
    bars = ax.bar(
        x + offset,
        [model_results[m] for m in metrics_names],
        width,
        label=model_name,
        color=color,
        alpha=0.85,
        edgecolor='white'
    )
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.003,
            f'{bar.get_height():.3f}',
            ha='center', va='bottom',
            fontsize=8, fontweight='bold', color=color
        )

ax.set_ylim(0.88, 1.03)
ax.set_xticks(x)
ax.set_xticklabels(metrics_names, fontsize=13)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparaison des modèles\nNaïve Bayes / SVM / Régression Logistique / CamemBERT',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('comparaison_modeles.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Tableau récapitulatif ---
print("\n" + "=" * 65)
print("TABLEAU RÉCAPITULATIF")
print("=" * 65)
print(f"  {'Modèle':<26} {'Accuracy':>9} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print(f"  {'-' * 63}")
for model_name, (model_results, _) in models.items():
    print(f"  {model_name:<26} "
          f"{model_results['Accuracy']:>9.3f} "
          f"{model_results['Precision']:>10.3f} "
          f"{model_results['Recall']:>8.3f} "
          f"{model_results['F1']:>8.3f}")
print("=" * 65)